# 05 · AVA 단일 Episode FFT 재로딩
소스는 Git에서, 가중치·데이터는 Hugging Face 고정 revision에서 받습니다. 실행 시 SOURCE_COMMIT을 기록하고 재개할 때 SOURCE_REF에 지정하십시오.
이 로컬 노트북 변경은 기존 Drive 노트북과 실행 중인 Colab에 자동 반영되지 않습니다.


In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, tempfile
SOURCE_REPOSITORY = 'https://github.com/eastha10/VLADE.git'
SOURCE_REF = globals().get('SOURCE_REF', 'main')  # 재개 시 이전 SOURCE_COMMIT을 지정하십시오.
ROOT = Path('/content/vlade_work')
ROOT.mkdir(parents=True, exist_ok=True)
def stage_source(repository: str, scratch_root: Path, source_ref: str = "main") -> Path:
    """Resolve a ref once and return a clean checkout of that immutable commit.

    Existing checkouts are never updated: another process may be using them.
    Git checks object integrity, so no per-file SHA256 inventory is needed.
    """
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_./-]*", source_ref) or ".." in source_ref:
        raise ValueError("Invalid Git source reference")
    scratch_root = scratch_root.resolve()
    scratch_root.mkdir(parents=True, exist_ok=True)

    def git(directory, *args):
        return subprocess.run(["git", "-C", str(directory), *args], check=True,
                              capture_output=True, text=True).stdout.strip()

    with tempfile.TemporaryDirectory(prefix="vlade-fetch-", dir=scratch_root) as temporary:
        staging = Path(temporary) / "source"
        git(scratch_root, "clone", "--no-checkout", "--filter=blob:none", "--", repository, str(staging))
        git(staging, "fetch", "--depth", "1", "origin", source_ref)
        commit = git(staging, "rev-parse", "FETCH_HEAD^{commit}")
        # Keep submodule metadata paths short on Windows; check the full SHA below.
        destination = scratch_root / ("vlade_colab_" + commit[:16])
        if destination.exists():
            if (git(destination, "rev-parse", "HEAD") != commit
                    or git(destination, "remote", "get-url", "origin") != repository
                    or git(destination, "status", "--porcelain", "--untracked-files=normal")
                    or any(line[:1] in {"-", "+", "U"} for line in git(destination, "submodule", "status").splitlines()
                           if "third_party/ava_vla" in line)):
                raise ValueError("Existing source checkout changed; use a fresh source cache")
            return destination
        git(staging, "checkout", "--detach", commit)
        # The parent commit pins the AVA gitlink. Other baselines use their own environments.
        if (staging / ".gitmodules").is_file():
            git(staging, "submodule", "update", "--init", "--depth", "1", "--", "third_party/ava_vla")
        staging.rename(destination)
    return destination

CODE = stage_source(SOURCE_REPOSITORY, Path('/content/vlade_sources'), SOURCE_REF)
PROJECT_ROOT = CODE
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip()
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('실행 소스:', CODE, '\nGit commit:', SOURCE_COMMIT)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'), HF_TOKEN_PATH='/content/vlade_hf_token')
def run(args, env=None):
    import signal, threading, time
    deadline = float((env or ENV).get('VLADE_DEADLINE_UNIX', '0'))
    if deadline and deadline - time.time() <= 60:
        raise RuntimeError('8시간 예산이 끝났습니다. 새 작업을 시작하지 않습니다.')
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timers = []
    if deadline:
        for delay, callback in [(max(0, deadline-time.time()-60), lambda: process.send_signal(signal.SIGINT)),
                                (max(0, deadline-time.time()), process.kill)]:
            timer = threading.Timer(delay, callback)
            timer.daemon = True
            timer.start()
            timers.append(timer)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('실행에 실패했거나 시간 예산이 끝났습니다. 위 결과를 확인하십시오.')
    finally:
        for timer in timers:
            timer.cancel()
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', 'uv'])
UV = shutil.which('uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

# Git supplies source. HF supplies every external model/data input.
ASSET_CACHE = Path('/content/vlade_hf_assets')
ASSET_WORK_ROOT = Path('/content/vlade_asset_work')
RUNS_ROOT = Path('/content/vlade_runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
_HF_FETCHED = set()
ENV.update(VLADE_ASSET_CACHE=str(ASSET_CACHE),
           VLADE_ASSET_REGISTRY=str(CODE/'configs/storage/hf_assets.lock.json'),
           HF_HOME='/content/vlade_hf_home', HF_TOKEN_PATH='/content/vlade_hf_token')
os.environ.update({key: ENV[key] for key in ('VLADE_ASSET_CACHE','VLADE_ASSET_REGISTRY')})

def hf_input(key, python=None, *, fetch=True):
    from vlade.storage.assets import asset_record, cache_path
    record = asset_record(key, Path(ENV['VLADE_ASSET_REGISTRY']))
    target = cache_path(key, record, ASSET_CACHE)
    if fetch and (key, str(target)) not in _HF_FETCHED:
        if python is None:
            raise ValueError('Select the CPU/model Python environment before fetching HF assets')
        run([python, 'scripts/data/hf_assets.py', 'fetch', key,
             '--registry', ENV['VLADE_ASSET_REGISTRY'], '--cache', ASSET_CACHE], env=ENV)
        _HF_FETCHED.add((key, str(target)))
    return target

def hf_path(value, python=None, *, fetch=True):
    value = str(value)
    if not value.startswith('hfasset:'):
        raise ValueError('External inputs must use hfasset:// keys, not Drive/local paths')
    return hf_input(value[len('hfasset:'):].strip('/\\'), python, fetch=fetch)

In [ ]:
# HF 설정 후 기존 학습 셀을 실행하면 checkpoint 저장 시 자동 전송합니다.
# 저장소는 HF 웹에서 Private로 먼저 만들고 해당 저장소 쓰기 토큰을 Secrets에 등록하십시오.
if 'ENV' not in globals():
    ENV = os.environ.copy()
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
ENV['HF_TOKEN'] = os.environ['HF_TOKEN']
import datetime, uuid
ENABLE_HUB_UPLOAD = globals().get('ENABLE_HUB_UPLOAD', True)
HF_REPO_ID = 'VLADE-konkuk/VLADE-checkpoints'  # 조직 저장소; 이전 런타임 값도 갱신합니다.
HF_RUN_GROUP = globals().get('HF_RUN_GROUP', datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])  # 재개 시 기존 group 지정
if ENABLE_HUB_UPLOAD:
    if not HF_RUN_GROUP:
        raise ValueError('HF_RUN_GROUP을 지정하십시오. pilot/main은 출력 폴더명으로 구분합니다.')
    os.environ['VLADE_HUB_REPO'] = HF_REPO_ID
    os.environ['VLADE_HUB_RUN_ID'] = HF_RUN_GROUP
    os.environ['HF_TOKEN_PATH'] = '/content/vlade_hf_token'
    # 모델 캐시는 기존 위치를 유지하고 토큰 파일은 런타임 로컬에만 둡니다.
    ENV.update(HF_TOKEN=os.environ['HF_TOKEN'], VLADE_HUB_REPO=HF_REPO_ID,
               VLADE_HUB_RUN_ID=HF_RUN_GROUP, HF_TOKEN_PATH='/content/vlade_hf_token')
    print('Private Hub 업로드 설정:', HF_REPO_ID, HF_RUN_GROUP)
else:
    for key in ('VLADE_HUB_REPO', 'VLADE_HUB_RUN_ID'):
        ENV.pop(key, None)
        os.environ.pop(key, None)
    print('Hub 업로드 꺼짐: HF_RUN_GROUP 설정 후 ENABLE_HUB_UPLOAD=True로 이 셀을 다시 실행하십시오.')

In [ ]:
AVA_ENV_READY = False
AVA_PY = None
import tempfile, tomllib, zipfile
AVA_VENV = Path('/content') / 'vlade_ava_fft_py311'
AVA_PATCH_REL = 'configs/models/teacher/transformers-avavla-a03eee5-checkpoint-extra-attn.patch'
AVA_PATCH_SHA = '9f0e46c9e93d62a92af950aeee712191cfcf314c1e636a9608fddcb57ff6087d'
AVA_TRANSFORMERS_REV = 'a03eee5da1794c45f5a911b60a9a4545e6e0ed1f'
AVA_BASE_SHA = '62f22856fe7b8459d6d61d4676b0075b65724979f84859ac475dbe36c10a3d80'
AVA_INSTALLED_SHA = '0a9c25300f3358f87af1719199608af3c27567dba0425690327fccd44cfb516b'
AVA_VERIFY_SOURCE = "import ast, hashlib, importlib, json, sys\nfrom pathlib import Path\nexpected_sha, expected_prefix = sys.argv[1:3]\nprefix = Path(sys.prefix).resolve()\nif sys.prefix == sys.base_prefix or prefix != Path(expected_prefix).resolve():\n    raise RuntimeError('AVA verification requires the dedicated virtual environment')\nmodule = importlib.import_module('transformers.models.llama.modeling_llama')\nsource = Path(module.__file__).resolve()\nif not source.is_relative_to(prefix) or source.name != 'modeling_llama.py':\n    raise RuntimeError('Transformers was imported outside the AVA environment')\npayload = source.read_bytes()\nif hashlib.sha256(payload).hexdigest() != expected_sha:\n    raise RuntimeError('Installed AVA Transformers source SHA mismatch')\ntree = ast.parse(payload)\nmodel = next(node for node in tree.body if isinstance(node, ast.ClassDef) and node.name == 'LlamaModel')\nforward = next(node for node in model.body if isinstance(node, ast.FunctionDef) and node.name == 'forward')\ncalls = [node for node in ast.walk(forward) if isinstance(node, ast.Call)\n         and isinstance(node.func, ast.Attribute) and node.func.attr == '_gradient_checkpointing_func']\nexpected = 'extra_attn_weights if extra_layer_ids and layer_idx in extra_layer_ids else None'\nif len(calls) != 1 or len(calls[0].args) != 9 or ast.unparse(calls[0].args[-1]) != expected:\n    raise RuntimeError('Installed checkpoint attention argument was not preserved')\nprint(json.dumps({'source': str(source), 'sha256': expected_sha, 'venv': str(prefix)}))"
_VLADE_AVA_BASE_RUN = getattr(run, '_vlade_base_run', run)

def verify_ava_transformers(python):
    _VLADE_AVA_BASE_RUN([python, '-c', AVA_VERIFY_SOURCE, AVA_INSTALLED_SHA, AVA_VENV])

def require_ava_environment():
    if not AVA_ENV_READY or AVA_PY is None:
        raise RuntimeError('AVA patched environment is not verified; rerun the installation cell')
    verify_ava_transformers(AVA_PY)

def run(args, env=None):
    command = [str(value) for value in args]
    if command and (command[0] == str(AVA_VENV / 'bin/python')
                    or any(Path(value).name == 'run_fft.py' for value in command)):
        require_ava_environment()
    return _VLADE_AVA_BASE_RUN(args, env=env)

run._vlade_base_run = _VLADE_AVA_BASE_RUN
patch = CODE / AVA_PATCH_REL
if not patch.is_file() or hashlib.sha256(patch.read_bytes()).hexdigest() != AVA_PATCH_SHA:
    raise RuntimeError('Staged AVA patch is missing or has the wrong SHA')

build_root = Path(tempfile.mkdtemp(prefix='vlade-ava-transformers-', dir=AVA_VENV.parent))
source = build_root / 'source'
source.mkdir()
build_env = ENV.copy()
build_env.update(UV_CACHE_DIR=str(build_root / 'uv-cache'), TMPDIR=str(build_root / 'tmp'))
(build_root / 'tmp').mkdir()
fork = 'https://github.com/LiAuto-DSR/transformers-avavla.git'
_VLADE_AVA_BASE_RUN(['git', '-C', source, 'init'], env=build_env)
_VLADE_AVA_BASE_RUN(['git', '-C', source, 'fetch', '--depth', '1', fork, AVA_TRANSFORMERS_REV], env=build_env)
_VLADE_AVA_BASE_RUN(['git', '-C', source, 'checkout', '--detach', 'FETCH_HEAD'], env=build_env)
revision = subprocess.check_output(['git', '-C', str(source), 'rev-parse', 'HEAD'], text=True).strip()
if revision != AVA_TRANSFORMERS_REV:
    raise RuntimeError('AVA Transformers checkout revision mismatch')
llama = source / 'src/transformers/models/llama/modeling_llama.py'
if hashlib.sha256(llama.read_bytes()).hexdigest() != AVA_BASE_SHA:
    raise RuntimeError('Unpatched AVA Transformers source SHA mismatch')
_VLADE_AVA_BASE_RUN(['git', '-C', source, 'apply', '--check', patch], env=build_env)
_VLADE_AVA_BASE_RUN(['git', '-C', source, 'apply', patch], env=build_env)
if hashlib.sha256(llama.read_bytes()).hexdigest() != AVA_INSTALLED_SHA:
    raise RuntimeError('Patched AVA Transformers source SHA mismatch')
wheels = build_root / 'wheels'
wheels.mkdir()
_VLADE_AVA_BASE_RUN([UV, 'build', '--wheel', '--python', '3.11', '--out-dir', wheels, source], env=build_env)
wheel_files = list(wheels.glob('transformers-*.whl'))
if len(wheel_files) != 1:
    raise RuntimeError('Expected exactly one patched Transformers wheel')
wheel = wheel_files[0]
with zipfile.ZipFile(wheel) as archive:
    wheel_source = archive.read('transformers/models/llama/modeling_llama.py')
if hashlib.sha256(wheel_source).hexdigest() != AVA_INSTALLED_SHA:
    raise RuntimeError('Built Transformers wheel does not contain the approved patch')

upstream = CODE / 'third_party/ava_vla'
dependencies = tomllib.loads((upstream / 'pyproject.toml').read_text())['project']['dependencies']
dependencies = [dep for dep in dependencies if not dep.startswith(('transformers @', 'dlimp @', 'huggingface_hub', 'accelerate'))]
dependencies += [
    str(wheel),
    'dlimp @ git+https://github.com/moojink/dlimp_openvla.git@040105d256bd28866cc6620621a3d5f7b6b91b46',
    'numpy==1.26.4', 'tensorflow-metadata==1.15.0', 'bitsandbytes==0.43.3',
    'huggingface_hub==0.36.2', 'accelerate==1.15.0']
candidate_python = environment('vlade_ava_fft_py311', '3.11', dependencies)
if str(candidate_python) != str(AVA_VENV / 'bin/python'):
    raise RuntimeError('Refusing to install outside the dedicated AVA environment')
_VLADE_AVA_BASE_RUN([UV, 'pip', 'install', '--python', candidate_python,
                    '--no-deps', '--force-reinstall', wheel], env=build_env)
_VLADE_AVA_BASE_RUN([UV, 'pip', 'install', '--python', candidate_python, '--no-deps', '-e', upstream])
verify_ava_transformers(candidate_python)
_VLADE_AVA_BASE_RUN([candidate_python, '-c', "import torch,transformers,tensorflow_datasets; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__,transformers.__version__,torch.cuda.get_device_name(0),torch.cuda.get_device_properties(0).total_memory/2**30)"])
freeze = subprocess.check_output([str(UV), 'pip', 'freeze', '--python', str(candidate_python)], text=True)
(ROOT / 'ava_environment_freeze.txt').write_text(freeze)
(AVA_VENV / 'vlade_transformers_patch.json').write_text(json.dumps({
    'base_commit': AVA_TRANSFORMERS_REV, 'patch_sha256': AVA_PATCH_SHA,
    'installed_source_sha256': AVA_INSTALLED_SHA,
    'wheel_sha256': hashlib.sha256(wheel.read_bytes()).hexdigest()}, indent=2))
AVA_PY = candidate_python
AVA_ENV_READY = True
print('AVA managed Transformers patch verified; FFT environment ready')

In [ ]:
TFDS_ROOT = hf_input('raw_rlds', AVA_PY)
PREPARED = ASSET_WORK_ROOT / 'data/processed/libero_plus_canonical_train1'
if not (PREPARED / 'COMPLETE').is_file():
    run([AVA_PY, 'scripts/data/prepare_single_canonical_episode.py',
         '--tfds-root', TFDS_ROOT, '--output', PREPARED,
         '--canonical-manifest', CODE / 'experiments/manifests/libero_plus_rlds/canonical_d_seed42.jsonl.gz',
         '--canonical-sha256', '487d2be1039ba6a4facc3ba44785ba5159245dfe53462a1d5a41ba8230446e80',
         '--canonical-normalization', CODE / 'experiments/manifests/libero_plus_rlds/normalization_train_d_seed42.json',
         '--canonical-normalization-sha256', 'eac95bcc81b6182b90a2bd030096df6e3e408cb88923d0a1d57dc6c81e297c67'])
manifest = json.loads((PREPARED / 'manifest.json').read_text())
assert manifest['dataset_scope'] == 'canonical_train_episode_smoke'
assert manifest['counts'] == {'train': 1}
assert manifest['canonical_manifest_sha256'] == '487d2be1039ba6a4facc3ba44785ba5159245dfe53462a1d5a41ba8230446e80'
assert manifest['canonical_normalization_sha256'] == 'eac95bcc81b6182b90a2bd030096df6e3e408cb88923d0a1d57dc6c81e297c67'
print(json.dumps({key: manifest[key] for key in (
    'dataset_scope', 'counts', 'selected_source_episode_index',
    'canonical_manifest_sha256', 'canonical_normalization_sha256')}, indent=2))
print('Episode:', manifest['episodes'][0]['episode_id'],
      'queries:', manifest['episodes'][0]['queries'])

In [ ]:
OUTPUT = RUNS_ROOT / 'ava_fft/ava-a100-one-step-seed42'
CONFIGS = RUNS_ROOT / 'run-configs'; CONFIGS.mkdir(exist_ok=True)
CONFIG = CONFIGS / 'ava-a100-one-step-seed42.json'
config = json.loads((CODE / 'configs/experiments/libero_plus_fft.json').read_text())
config['fft_defaults'].update(
    data_root=str(TFDS_ROOT), prepared_data_root=str(PREPARED), image_aug=False,
    batch_size=1, grad_accumulation_steps=1, max_steps=1, save_steps=1,
    save_optimizer_state=True, eval_steps=0, debug_records=0)
config['fft_runs']['ava_vla'].update(
    checkpoint=str(hf_input('ava_t0', AVA_PY)),
    output_dir=str(OUTPUT))
if CONFIG.exists() and json.loads(CONFIG.read_text()) != config:
    raise ValueError('기존 1-step 설정이 다릅니다. OUTPUT/CONFIG 이름을 바꾸십시오.')
CONFIG.write_text(json.dumps(config, indent=2))
run([AVA_PY, 'scripts/train/run_fft.py', '--model', 'ava_vla',
     '--config', CONFIG, '--preflight'])

In [ ]:
CHECKPOINT = OUTPUT / 'step-0000001'
if not (CHECKPOINT / 'COMPLETE').is_file():
    run([AVA_PY, 'scripts/train/run_fft.py', '--model', 'ava_vla', '--config', CONFIG])
metrics = sorted(OUTPUT.glob('metrics-*.jsonl'))
runs = sorted(OUTPUT.glob('run-*.json'))
if len(metrics) != 1 or len(runs) != 1:
    raise ValueError('격리된 새 OUTPUT에는 metrics/run manifest가 각각 하나여야 합니다.')
run([AVA_PY, 'scripts/train/check_fft_checkpoint.py',
     '--checkpoint', CHECKPOINT, '--data', PREPARED, '--config', CONFIG,
     '--metrics', metrics[0], '--run-manifest', runs[0],
     '--report', OUTPUT / 'reload_report.json'])

In [ ]:
report = json.loads((OUTPUT / 'reload_report.json').read_text())
summary = {
    'status': report['status'], 'gpu': report['gpu'],
    'step_time_seconds': report['training_metrics']['step_time_seconds'],
    'training_peak_allocated_gib': report['training_metrics']['peak_allocated_bytes'] / 2**30,
    'training_peak_reserved_gib': report['training_metrics']['peak_reserved_bytes'] / 2**30,
    'training_peak_process_ram_gib': report['training_metrics']['peak_process_rss_bytes'] / 2**30,
    'reload_peak_allocated_gib': report['reload_peak_allocated_bytes'] / 2**30,
    'reload_peak_process_ram_gib': report['reload_peak_process_rss_bytes'] / 2**30,
    'checkpoint_gib': report['checkpoint_bytes'] / 2**30,
    'loss': report['training_metrics']['loss'],
    'grad_norm': report['training_metrics']['grad_norm'],
    'gradient_diagnostics': report['gradient_diagnostics'],
    'action_finite': report['action_finite'],
    'training_finite_checks': report['training_finite_checks'],
    'scope': report['scope'],
    'ava_temporal_training': 'zero_context',
}
print(json.dumps(summary, indent=2))
print('원본 보고서:', OUTPUT / 'reload_report.json')